In [ ]:
import librosa
import numpy as np
import os
import pandas as pd
from tqdm import tqdm #tracking progress

In [36]:
def extract_audio_features(file_path, task='mood', sr=22050):
    try:
        #Load audio file
        y, sr = librosa.load(file_path, sr=sr)

        #Extract features
        features = {

            #MFCC (Mel-frequency cepstral coefficients) - Captures timbre
            "mfcc_std": np.std(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13 if task == 'mood' else 20), axis=1),
            "mfcc_mean": np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13 if task == 'mood' else 20,), axis=1),

            #Spectral features; perceived brightness of sounds
            "spectral_centroid": np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
            "spectral_bandwidth": np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)),
            "spectral_contrast": np.mean(librosa.feature.spectral_contrast(y=y, sr=sr), axis=1),
            "spectral_rolloff": np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)),

            #Chroma features - harmonic patterns and pitch content
            "chroma_stft_mean": np.mean(librosa.feature.chroma_stft(y=y, sr=sr)),
            
            #Rms
            "rms_mean": np.mean(librosa.feature.rms(y=y)),
            
            #Zero Crossing Rate - Measures noisiness
            "zcr": np.mean(librosa.feature.zero_crossing_rate(y=y)),

            #Mel-Spectrogram
            "mel_spec_mean": np.mean(librosa.feature.melspectrogram(y=y, sr=sr), axis=1),
        }

        if task == 'instrument':
            #Harmonic-to-Noise Ration (HNR)
            features['hnr_mean'] = np.mean(librosa.effects.harmonic(y) / (librosa.effects.percussive(y) + 1e-6))
            #Onset Strength - Intesity of note onsets (sharp changed in energy)
            features['onset_strength'] = np.mean(librosa.onset.onset_strength(y=y, sr=sr))

        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        if task == 'genre' or task == 'mood':
            #Tempo - Beats Per Minute 
            features['tempo'] = tempo

        if task == 'mood':
            #Dynamics - variability in loudness
            features['rms_std'] = np.std(librosa.feature.rms(y=y))

        return features #Return dictionary of extracted features
    except Exception as e:
        print(f"Error processing {dataset_path}: {e}")
        return None

In [ ]:
#Process EmoMusic dataset, extracting features and linking them to mood labels.
def process_filtered_emomusic(audio_dir, song_info_file):

    # Load song info (only annotated files)
    song_info = pd.read_csv(song_info_file)
    data = []  # To store features and labels

    for _, row in tqdm(song_info.iterrows(), total=len(song_info), desc="Processing EmoMusic"):
        song_id = str(row['song_id'])
        file_name = row['file_name']
        start_of_the_segment = row['start of the segment (min.sec)']
        end_of_the_segment = row['end of the segment (min.sec)']
        Genre = row['Genre']
        Mediaeval_2013_Set = row['Mediaeval 2013 set']
        mean_arousal = row['mean_arousal']
        std_arousal = row['std_arousal']
        mean_valence = row['mean_valence']
        std_valence = row['std_valence']

        # Locate audio file by song ID
        file_path = os.path.join(audio_dir, f"{song_id}.mp3")
        if os.path.exists(file_path):
            features = extract_audio_features(file_path, 'mood')
            if features:
                # Add features to data
                data.append({
                    "song_id": song_id,
                    "file_name": file_name,
                    "start_of_the_segment": start_of_the_segment,
                    "end_of_the_segment": end_of_the_segment,
                    "Genre": Genre,
                    "Mediaeval_2013_set": Mediaeval_2013_Set,
                    "mean_arousal": mean_arousal,
                    "std_arousal": std_arousal,
                    "mean_valence": mean_valence,
                    'std_valence': std_valence,
                    **features
                })
        else:
            print(f"Audio file not found for song ID: {song_id}")

        # Flatten feature dictionaries
    processed_data = []
    for feat in data:
        flat_feat = {key: feat[key] for key in ["song_id", "file_name", "start_of_the_segment", "end_of_the_segment", "Genre", "Mediaeval_2013_set", "mean_arousal", "std_arousal", "mean_valence", "std_valence", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff", "chroma_stft_mean", "rms_mean", "rms_std", "zcr", "tempo"]}  # Keep unflattened
        flat_feat.update({f'spectral_contrast_{i}': feat['spectral_contrast'][i] for i in range(len(feat['spectral_contrast']))})
        flat_feat.update({f'mfcc_mean_{i}': feat['mfcc_mean'][i] for i in range(len(feat['mfcc_mean']))})
        flat_feat.update({f'mfcc_std_{i}': feat['mfcc_std'][i] for i in range(len(feat['mfcc_std']))})
        flat_feat.update({f'mel_spec_mean_{i}': feat['mel_spec_mean'][i] for i in range(len(feat['mel_spec_mean']))})
        processed_data.append(flat_feat)

    return pd.DataFrame(processed_data)


In [ ]:

# Example usage
audio_dir = r"./data/rawEMOMUSIC - clips_45seconds/Audio"
song_info_file = r"./data/processed/EMOMUSIC_merged_info.csv"

# Process dataset
audio_features_df = process_filtered_emomusic(audio_dir, song_info_file)

unflattened_columns = ["song_id", "file_name", "start_of_the_segment", "end_of_the_segment", "Genre", "Mediaeval_2013_set", "mean_arousal", "std_arousal", "mean_valence", "std_valence", "chroma_stft_mean", "rms_mean", "rms_std", "zcr", "tempo", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff"]
flattened_columns = [col for col in audio_features_df.columns if col not in unflattened_columns]

#unflattened + flattened columns
audio_features_df = audio_features_df[unflattened_columns + flattened_columns]

# Save to CSV
audio_features_df.to_csv("./data/processed/EMOMUSIC_features_45_sec.csv", index=False)
print("Saved EmoMusic features to './data/processed/EMOMUSIC_features_45_Sec.csv'")

Processing EmoMusic:   0%|          | 0/744 [00:00<?, ?it/s]

Processing EmoMusic: 100%|██████████| 744/744 [15:41<00:00,  1.27s/it]


Saved EmoMusic features to 'EMOMUSIC_features_45_Sec.csv'


In [30]:
#Process dataset, extracting features for dataframe
def process_dataset(dataset_path):
    data = [] #stores features and labels
    for label in os.listdir(dataset_path):
        label_path = os.path.join(dataset_path, label)
        if os.path.isdir(label_path): #Check if folder
            for file in tqdm(os.listdir(label_path), desc=f"Processing {label}"):
                file_path = os.path.join(label_path, file)
                if file_path.endswith(".wav"): #Process only .wav files
                    file_name, _ = os.path.splitext(os.path.basename(file_path))
                    features = extract_audio_features(file_path)
                    if features:
                        #Add filename, label, and features to dataset
                        data.append({"file_name": file_name, "label": label, **features})
    
    # Flatten feature dictionaries
    processed_data = []
    for feat in data:
        flat_feat = {key: feat[key] for key in ["file_name", "label", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff", "chroma_stft_mean", "rms_mean", "zcr", "hnr_mean", "onset_strength"]}  # Keep unflattened
        flat_feat.update({f'spectral_contrast_{i}': feat['spectral_contrast'][i] for i in range(len(feat['spectral_contrast']))})
        flat_feat.update({f'mfcc_mean_{i}': feat['mfcc_mean'][i] for i in range(len(feat['mfcc_mean']))})
        flat_feat.update({f'mfcc_std_{i}': feat['mfcc_std'][i] for i in range(len(feat['mfcc_std']))})
        flat_feat.update({f'mel_spec_mean_{i}': feat['mel_spec_mean'][i] for i in range(len(feat['mel_spec_mean']))})
        processed_data.append(flat_feat)

    #Convert to Pandas Dataframe
    return pd.DataFrame(processed_data)

In [ ]:
#Path to my unzipped dataset
dataset_path = r".\data\IRMAS-TrainingData"

#Process dataset
audio_features_df = process_dataset(dataset_path)

#Order Columns
unflattened_columns = ["file_name", "label", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff", "chroma_stft_mean", "rms_mean", "zcr", "hnr_mean", "onset_strength"]
flattened_columns = [col for col in audio_features_df.columns if col not in unflattened_columns]

#unflattened + flattened columns
audio_features_df = audio_features_df[unflattened_columns + flattened_columns]

#Save features to a csv for future use
audio_features_df.to_csv("IRMAS_features_2_sec.csv", index=False)

print("Feature extraction complete! Data saved to 'irmas_features_2_sec.csv'.")

Processing voice: 100%|██████████| 778/778 [06:47<00:00,  1.91it/s]


Feature extraction complete! Data saved to 'irmas_features.csv'.
